## Idea

This notebook trains the baseline xgb model using SG5F cross validation

---

### Imports

In [1]:
from pathlib import Path

import mlflow
import mlflow.xgboost
import mlflow.data

import numpy as np
import pandas as pd

import xgboost as xgb

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score

### Paths + mlflow

In [ ]:
PROJECT_ROOT = Path.cwd().parent.parent

DATA_PATH = PROJECT_ROOT / "data"

TRAIN_PATH = DATA_PATH / "dvc" / "train_v3.parquet"

MLRUNS_PATH = PROJECT_ROOT / "mlruns"

mlflow.set_tracking_uri(f"file:{MLRUNS_PATH}")
mlflow.set_experiment("simbox_fraud_detection")

c:\Users\ali.farhat\AppData\Local\miniconda3\envs\simbox\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:c:\\Users\\ali.farhat\\Documents\\simbox-project\\mlruns/181740959998191728', creation_time=1781294983928, experiment_id='181740959998191728', last_update_time=1781294983928, lifecycle_stage='active', name='simbox_fraud_detection', tags={}, trace_location=None, workspace='default'>

### Load dataset

In [3]:
df = pd.read_parquet(TRAIN_PATH)

print(df.shape)
display(df.head())

print("\nLabel distribution:")
print(df["label"].value_counts(normalize=True))

print("\nUnique MSISDNs:", df["msisdn"].nunique())
print("Unique days:", df["profile_date"].nunique())


# Build features
X = df.drop(columns=["label", "msisdn", "profile_date"])
y = df["label"]
groups = df["msisdn"]

feature_names = X.columns.tolist()

print(X.shape)

(343040, 15)


,msisdn,profile_date,label,repeat_call_ratio,flagged_outgoing_ratio,incoming_outgoing_ratio,incoming_calls,unique_callees,avg_outgoing_duration,flagged_calls,short_call_ratio,unique_callers,avg_incoming_duration,avg_outgoing_calls_per_hour,peak_hour_ratio
0,22362444330,2022-11-01,0,0.800000,999999.0,999999.0,5,0,0.0,5,0.000000,4,136.200000,0.000000,0.4
1,22667898029,2022-11-01,0,0.000000,999999.0,999999.0,1,0,0.0,1,0.000000,1,90.000000,0.000000,1.0
2,22995468560,2022-11-01,0,0.500000,999999.0,999999.0,2,0,0.0,2,0.500000,1,38.000000,0.000000,1.0
3,2250101000087,2022-11-01,0,0.400000,2.5,1.5,3,2,57.5,5,0.600000,3,23.666667,2.000000,0.6
4,2250101000108,2022-11-01,0,0.166667,1.2,0.2,2,9,329.1,12,0.333333,2,40.000000,3.333333,0.5



Label distribution:
label
0    0.942499
1    0.057501
Name: proportion, dtype: float64

Unique MSISDNs: 25394
Unique days: 15
(343040, 12)


### SGKF setup + imbalance

In [4]:
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scale_pos_weight = (y == 0).sum() / (y == 1).sum()

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 16.39112801013942


### Model params

In [5]:
params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "eta": 0.05,
    "max_depth": 6,
    "min_child_weight": 1,
    "subsample": 1,
    "colsample_bytree": 1,
    "scale_pos_weight": scale_pos_weight,
    "seed": 42,
    "n_jobs": -1
}

### SGKF training + OOF predictions

In [6]:
fold_results = []
oof_preds = np.zeros(len(X))
models = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups=groups)):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=300
    )

    preds = model.predict(dval)

    oof_preds[val_idx] = preds

    pr_auc = average_precision_score(y_val, preds)
    roc_auc = roc_auc_score(y_val, preds)

    fold_results.append({
        "fold": fold,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc
    })

    models.append(model)

    print(f"Fold {fold}: PR-AUC={pr_auc:.5f} ROC-AUC={roc_auc:.5f}")

Fold 0: PR-AUC=0.42511 ROC-AUC=0.86464
Fold 1: PR-AUC=0.41863 ROC-AUC=0.86667
Fold 2: PR-AUC=0.43489 ROC-AUC=0.86604
Fold 3: PR-AUC=0.44920 ROC-AUC=0.87271
Fold 4: PR-AUC=0.42051 ROC-AUC=0.86454


### Log run to mlflow

In [7]:
with mlflow.start_run(run_name="sgkf_baseline_v1"):

    # ---- dataset logging ----
    mlflow.log_param("train_path", str(TRAIN_PATH))
    mlflow.log_param("n_features", len(feature_names))
    mlflow.log_param("n_samples", len(df))
    mlflow.log_param("fraud_ratio", float(y.mean()))
    mlflow.log_param("n_msisdn", df["msisdn"].nunique())

    # ---- model params ----
    mlflow.log_params(params)

    # ---- fold metrics ----
    pr_list = [r["pr_auc"] for r in fold_results]
    roc_list = [r["roc_auc"] for r in fold_results]

    for i, r in enumerate(fold_results):
        mlflow.log_metric(f"fold_{i}_pr_auc", r["pr_auc"])
        mlflow.log_metric(f"fold_{i}_roc_auc", r["roc_auc"])

    mlflow.log_metric("mean_pr_auc", np.mean(pr_list))
    mlflow.log_metric("std_pr_auc", np.std(pr_list))

    mlflow.log_metric("mean_roc_auc", np.mean(roc_list))
    mlflow.log_metric("std_roc_auc", np.std(roc_list))

    # ---- log OOF predictions ----
    oof_df = pd.DataFrame({
        "msisdn": df["msisdn"],
        "profile_date": df["profile_date"],
        "label": y,
        "oof_pred": oof_preds
    })

    oof_path = DATA_PATH / "oof_predictions_v3.parquet"
    oof_df.to_parquet(oof_path, index=False)

    mlflow.log_artifact(str(oof_path), artifact_path="oof_predictions")

    # ---- log one model ----
    mlflow.xgboost.log_model(models[0], artifact_path="model")

2026/07/05 22:09:57 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/07/05 22:09:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instea

### Results table

In [8]:
results_df = pd.DataFrame(fold_results)

display(results_df)

print("\nMean PR-AUC:", results_df["pr_auc"].mean())
print("Std PR-AUC:", results_df["pr_auc"].std())

print("\nMean ROC-AUC:", results_df["roc_auc"].mean())
print("Std ROC-AUC:", results_df["roc_auc"].std())

,fold,pr_auc,roc_auc
0,0,0.425112,0.864637
1,1,0.418628,0.866672
2,2,0.434894,0.866041
3,3,0.449200,0.872705
4,4,0.420509,0.864536



Mean PR-AUC: 0.4296686404290959
Std PR-AUC: 0.012602996489026193

Mean ROC-AUC: 0.866918339829037
Std ROC-AUC: 0.0033614262223707623


### OOF summary

In [9]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y, oof_preds)

print("OOF PR-AUC:", average_precision_score(y, oof_preds))
print("OOF ROC-AUC:", roc_auc_score(y, oof_preds))

OOF PR-AUC: 0.4294247033268884
OOF ROC-AUC: 0.8668627731959323
